In [27]:
import requests
import pandas as pd
import numpy as np
import json

# 1. Initial Inspection of the Dataset

In [28]:
url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Cell_Phones_and_Accessories.jsonl"
records = []
with requests.get(url,stream=True) as r:
    for i,line in enumerate(r.iter_lines()):
        if i>200000:
            break
        if line:
            records.append(json.loads(line.decode("utf-8")))

df = pd.DataFrame(records)

## Data Loading — Issue & Fix

**Problem:**
The standard `load_dataset()` from HuggingFace failed because the Amazon Reviews 2023
dataset uses a custom loading script (`Amazon-Reviews-2023.py`) which is no longer
supported by newer versions of the `datasets` library.

**What we tried:**
- `load_dataset()` with `trust_remote_code=True` → blocked by library version
- `load_dataset()` with `streaming=True` → same error

**Fix:**
Bypassed HuggingFace loader entirely. Used `requests` to stream the raw `.jsonl` file
directly from the HuggingFace CDN, reading 200K rows line by line without downloading
the full 9.34GB file.

**Why streaming?**
The full Cell Phones & Accessories review file is 9.34GB. Streaming reads one line at a
time and stops at 200K rows — no unnecessary data downloaded.

**Why `json.loads()` instead of `pd.read_json()`?**
Each streamed line arrives as `bytes`. I decoded it to a string with `.decode("utf-8")`,
then parse it into a Python dict with `json.loads()`. `pd.read_json()` expects a file
path or file-like object, not a raw string.

In [29]:
print(df.shape)
print(df.columns)
print(df.head())

(200001, 10)
Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='object')
   rating                                title  \
0     4.0     No white background! It’s clear!   
1     5.0  Awesome!  Great price!  Works well!   
2     5.0   Worked but took an hour to install   
3     4.0                               Decent   
4     5.0                             LOVE IT!   

                                                text  \
0  I bought this bc I thought it had the nice whi...   
1  Perfect. How pissed am I that I recently paid ...   
2  Overall very happy with the end result. If you...   
3  Lasted about 9 months then the lock button bro...   
4  LOVE THIS CASE! Works better than my expensive...   

                                              images        asin parent_asin  \
0  [{'small_image_url': 'https://images-na.ssl-im...  B08L6L3X1S  B08L6L3X1S   
1                              

In [30]:
df_filtered = df.drop(columns=['images','parent_asin','user_id',])

In [31]:
df_filtered = df_filtered.drop(df_filtered[df_filtered['rating']>2.0].index)
print("After rating filter: ",df_filtered.shape)

After rating filter:  (30197, 7)


In [32]:
df_filtered = df_filtered.dropna(subset=['text'])
print("After dropping Nulls: ", df_filtered.shape)

After dropping Nulls:  (30197, 7)


In [33]:
print(df_filtered['rating'].value_counts())
print(df_filtered['text'].str.split().str.len().describe())
print(df_filtered.head(3)[['rating','title','text']])

rating
1.0    19261
2.0    10936
Name: count, dtype: int64
count    30197.000000
mean        47.191906
std         62.406626
min          0.000000
25%         15.000000
50%         30.000000
75%         57.000000
max       1729.000000
Name: text, dtype: float64
    rating                                              title  \
16     2.0                       Don't try to tighten it up!!   
27     2.0  A bit complicated to apply. Requires spraying ...   
30     1.0                                     Buyer BEWARE!!   

                                                 text  
16  Putting it on a night stand drawer or top of h...  
27  A bit complicated to apply. Requires spraying ...  
30  This product is cheaply made, some of the part...  


In [38]:
print(df_filtered['text'][df_filtered['text'].str.split().str.len()==0].count())

14


In [39]:
df_filtered = df_filtered[df_filtered['text'].str.split().str.len()!=0]

In [40]:
print(df_filtered['text'].str.split().str.len().min())

1
